# 24. Однократный финальный global test

Запускайте только после фиксации NER checkpoint, RE checkpoint и обоих validation-порогов. Notebook считает: NER component, RE component на gold-сущностях и полный NER→RE pipeline.

In [ ]:
RUN_FINAL_GLOBAL_TEST = False  # Осознанно измените на True только один раз после freeze.
if not RUN_FINAL_GLOBAL_TEST:
    raise RuntimeError('Global test закрыт. Сначала завершите notebooks 18–23, затем установите RUN_FINAL_GLOBAL_TEST=True.')


In [ ]:
from pathlib import Path
import json, os, runpy
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass
PROJECT_DIR = Path('/content/drive/MyDrive/NER_RuREBus_project')
if not PROJECT_DIR.exists(): PROJECT_DIR = Path.cwd()
os.environ['HF_HOME'] = '/content/huggingface_cache'
runpy.run_path(str(PROJECT_DIR / 'colab_bootstrap.py'))['bootstrap_project'](PROJECT_DIR)


In [ ]:
from rurebus_ie.training import test_hierarchical_span_ner_experiment, evaluate_relation_experiment, test_end_to_end_pipeline_experiment
NER_CONFIG = PROJECT_DIR / 'configs/experiments/hierarchical_span_ner_global_v1.yaml'
RE_CONFIG = PROJECT_DIR / 'configs/experiments/relation_classifier_global_v1.yaml'
NER_RUN = PROJECT_DIR / 'results/hierarchical_span_ner_global_v1/seed_42'
RE_RUN = PROJECT_DIR / 'results/relation_classifier_global_v1/seed_42'
def read_json(path):
    with path.open(encoding='utf-8') as stream: return json.load(stream)
NER_THRESHOLD = float(read_json(NER_RUN / 'threshold_calibration.json')['best_threshold'])
RE_THRESHOLD = float(read_json(RE_RUN / 'relation_threshold_calibration.json')['best_threshold'])
PIPELINE_THRESHOLD = float(read_json(RE_RUN / 'pipeline_threshold_calibration.json')['best_threshold'])
print('Frozen thresholds:', {'ner': NER_THRESHOLD, 'relation_gold': RE_THRESHOLD, 'pipeline_relation': PIPELINE_THRESHOLD})


In [ ]:
ner = test_hierarchical_span_ner_experiment(NER_CONFIG, project_root=PROJECT_DIR, confidence_threshold_override=NER_THRESHOLD, artifact_prefix='global_test')
relation = evaluate_relation_experiment(RE_CONFIG, split_key='test', project_root=PROJECT_DIR, confidence_threshold_override=RE_THRESHOLD, artifact_prefix='global_test_gold_entities')
pipeline = test_end_to_end_pipeline_experiment(RE_CONFIG, ner_test_predictions_path=NER_RUN / 'global_test_predictions.jsonl', confidence_threshold=PIPELINE_THRESHOLD, project_root=PROJECT_DIR)
summary = {
    'protocol': 'global_v1',
    'frozen_thresholds': {'ner': NER_THRESHOLD, 'relation_gold': RE_THRESHOLD, 'pipeline_relation': PIPELINE_THRESHOLD},
    'ner': ner.metrics.to_dict(),
    'relation_gold_entities': relation.metrics.to_dict(),
    'relation_candidate_recall': relation.candidate_recall,
    'pipeline_candidate_recall': pipeline.candidate_recall,
    'end_to_end_pipeline': pipeline.metrics.to_dict(),
}
with (RE_RUN / 'final_global_test_summary.json').open('w', encoding='utf-8') as stream:
    json.dump(summary, stream, ensure_ascii=False, indent=2)
for name, metrics in [('NER', ner.metrics), ('RE / gold entities', relation.metrics), ('Full pipeline', pipeline.metrics)]:
    print(f"{name:20s} micro-F1={metrics.micro_f1:.6f} macro-F1={metrics.macro_f1:.6f} P={metrics.precision:.6f} R={metrics.recall:.6f}")
